# 02 — the GPU path, on a machine that may not have a GPU

Every cell here runs to completion whether or not an adapter is present. If
`wgpu` cannot find one, the notebook **says so plainly** and carries on with the
CPU path; it does not fail and it does not pretend.

Two things are established before any comparison is made:

* the current API does **not** provide a GPU-resident `Table`/`Series` —
  operation results are returned to host memory;
* the GPU produces the **same computed results**, not a faster answer.

Neither is a caveat added here. Both are pinned by
`tests/fast/test_doc_gpu_residency.py`, and this notebook is scanned by that
same guard — markdown, source **and outputs** — for a residency claim.

In [1]:
import importlib.util
import sys

import numpy as np
import numfast as nf

print("python ", sys.version.split()[0])
print("numfast", nf.__version__)

python  3.14.6
numfast 0.2.1


## 1. Is the native path live?

`native_info()` is the honest answer to "is the native DLL actually here?".
On Linux and macOS it reports `dll: None` — absent by construction, not broken.
This is the CPU-side disclosure; the GPU is separate.

In [2]:
info = nf.native_info()
for key in sorted(info):
    print(f"  {key:11s} {info[key]}")

  disabled    False
  dll         C:\App\numfast\numfast\src\numfast\_native\numfast_native.dll
  dll_exists  True


## 2. Adapter detection — attempted live, never assumed

`gpu_capabilities()` reports what the engine *supports*. It does not report
whether an adapter is *present*. Those are different questions, and conflating
them is how a "GPU: yes" gets printed on a laptop with no GPU, so the adapter is
probed directly through `wgpu`, the same library the driver uses.

In [3]:
HAVE_WGPU = importlib.util.find_spec("wgpu") is not None
print("wgpu importable:", HAVE_WGPU)

adapter = None
if HAVE_WGPU:
    try:
        import wgpu
        adapter = wgpu.gpu.request_adapter_sync(power_preference="high-performance")
    except Exception as err:                      # no adapter is a supported state
        print(f"no adapter: {type(err).__name__}: {err}")

if adapter is None:
    print("ADAPTER: none -- the GPU cells below will report unavailable")
else:
    info_ = adapter.info
    for field in ("vendor", "device", "description",
                  "adapter_type", "backend_type"):
        print(f"  {field:13s} {getattr(info_, field, '<absent>')}")

wgpu importable: True


  vendor        NVIDIA
  device        NVIDIA GeForce RTX 2060
  description   617.14
  adapter_type  <absent>
  backend_type  <absent>


A note on what that `description` field is: on this backend it carries the
driver version, not a marketing name. The `device` line is the one to read.

## 3. What `gpu_capabilities()` actually reports

`ops` is the set that executes on the device. Everything outside it executes on
the CPU and `cpu_only_ops` names each one. Requesting `backend='gpu'` for a
graph that uses a CPU-only operation **raises and names it** — there is no
silent fallback.

In [4]:
caps = nf.gpu_capabilities()
print("gpu ops      :", len(caps["ops"]))
print(caps["ops"])

gpu ops      : 15
['series', 'pack_keys', 'compare', 'mask', 'filter', 'gather', 'reduce', 'groupby', 'groupby_multi', 'sort', 'slice', 'shift', 'map', 'cumsum', 'rng_fill_i32']


In [5]:
full = nf.app().capabilities()
print("op_count       :", full["op_count"])
print("gpu_op_count   :", full["gpu_op_count"])
print("cpu_only count :", full["cpu_only_op_count"])
print("cpu_only ops   :", full["cpu_only_ops"])

op_count       : 33
gpu_op_count   : 15
cpu_only count : 18
cpu_only ops   : ['encode_pattern', 'group_count_distinct', 'lookup', 'map_round', 'rng_compat', 'rng_fill_f64', 'rng_permutation', 'rng_sample_no_replace', 'rolling_sum', 'text_contains', 'text_endswith', 'text_equals', 'text_length', 'text_regex_replace', 'text_startswith', 'unique', 'unique_inverse', 'where']


In [6]:
print("on a CPU-only op under backend='gpu':")
print(" ", full["on_cpu_only_under_gpu_backend"])
print("backend='auto':")
print(" ", full["backend_auto"])

on a CPU-only op under backend='gpu':
  asking for backend='gpu' on a graph that uses one of these raises RuntimeError naming 'op:<name>' -- there is no silent CPU fallback
backend='auto':
  backend='auto' resolves to the CPU unless a measured calibration reports a strictly lower GPU host cost; an unmeasured or uncovered graph resolves to the CPU, never to a fabricated GPU choice


## 4. The same task on the CPU and on the GPU

The consumer surface does not reach the GPU at all: `Chain.compile()` runs the
CPU path and `Chain.explain()` prints `backend=n/a`. That is a fact about this
API, and the `explain()` output below shows it rather than describing it.

The GPU is reachable at the **kernel level**, through `nf.get_kernel().alias`.
`gpu_execute` is a batch execute with per-node read-back.

The task is a filter plus a reduce: keep the values above a threshold, then sum
them. Both stages are GPU ops. Seed 42, fixed, so the two paths are comparable.

In [7]:
n = 1_000_000
rng = np.random.default_rng(42)
values = rng.integers(0, 1_000, n, dtype=np.int32)

t = nf.Table(nf.get_kernel(), {"v": nf.from_numpy(values, name="v")})
app = nf.app()

cpu_sum = t.query().filter(app.c("v") > 500).reduce("sum")
print("consumer surface (CPU only):")
print("  filter(v > 500).reduce('sum') ->", cpu_sum)

# explain() is where the consumer surface says, in its own output, that it has
# no backend choice at all.
line = [l for l in t.query().filter(app.c("v") > 500).explain().splitlines()
        if l.startswith("EXPLAIN")][0]
print("  explain() reports:", line)

consumer surface (CPU only):
  filter(v > 500).reduce('sum') -> 373949396.0
  explain() reports: EXPLAIN backend=n/a requested=n/a


In [8]:
alias = nf.get_kernel().alias
print("gpu_execute in the public alias table:", "gpu_execute" in alias)

# series -> compare -> filter: the same task the consumer surface just ran on the CPU.
jobs = [alias["ir_series"]("v", values),
        alias["ir_compare"]("m", "v", 500, ">"),
        alias["ir_filter"]("f", "v", "m")]
graph = alias["compile"](jobs)
print("compiled graph outputs:", graph["outputs"])
print("nodes:", [node["kernel_id"] for node in graph["nodes"]])

gpu_execute in the public alias table: True
compiled graph outputs: ['f']
nodes: ['series', 'compare', 'filter']


`compile()` here is the kernel-level planner call, producing **one** execution
graph for the whole job list — the fixed-cost property the README describes.

In [9]:
if adapter is not None:
    try:
        gpu_buffers = alias["gpu_execute"](graph["nodes"])
        kept = np.asarray(gpu_buffers["f"])
        print(f"GPU rows kept : {kept.size}")
        print(f"GPU sum       : {int(kept.astype(np.int64).sum())}")
        print("buffers returned (per-node read-back, NOT a resident handle):",
              sorted(gpu_buffers))
    except Exception as err:
        print(f"gpu_execute unavailable: {type(err).__name__}: {err}")
else:
    print("NO ADAPTER -- GPU path not attempted. Reporting CPU result only.")

GPU rows kept : 498843
GPU sum       : 373949396
buffers returned (per-node read-back, NOT a resident handle): ['f', 'm', 'v']


In [10]:
cpu_kept = values[values > 500]
print(f"CPU rows kept : {cpu_kept.size}")
print(f"CPU sum       : {int(cpu_kept.astype(np.int64).sum())}")
print("consumer-surface sum agrees with the CPU array:", cpu_sum == int(cpu_kept.astype(np.int64).sum()))
if adapter is not None and "gpu_buffers" in dir():
    print("GPU answer agrees with the CPU answer:",
          int(np.asarray(gpu_buffers["f"]).astype(np.int64).sum())
          == int(cpu_kept.astype(np.int64).sum()))

CPU rows kept : 498843
CPU sum       : 373949396
consumer-surface sum agrees with the CPU array: True
GPU answer agrees with the CPU answer: True


The two agree **exactly**. That is the honest GPU claim: same results, and
batch execution — not speed.

## 5. The crossover, from the repository's own recorded measurements

No wall-clock number is measured in this notebook. The box is shared, and a
timing taken here describes this box at this moment, not the engine.

`calibration_dataset.json` is in the tree, is a *measured* artefact (`source:
"measured:seed42"`, `reps: 5`, `warmup: 3`) and was produced on one RTX 2060 over
Vulkan. Every figure below is read from it. `04_benchmark.ipynb` uses the same
artefact.

In [11]:
import json
from pathlib import Path

ROOT = Path.cwd()
ds = json.loads((ROOT / "calibration_dataset.json").read_text(encoding="utf-8"))
prof = ds["profile"]
print("source   :", prof["source"])
print("generated:", prof["generated"])
for key in ("cpu", "gpu_device", "gpu_backend", "platform", "python"):
    print(f"  {key:11s} {prof['hardware'][key]}")
print("  reps      ", prof["measurements"]["reps"],
      " warmup", prof["measurements"]["warmup"],
      " Ns", prof["measurements"]["Ns"])

source   : measured:seed42
generated: 2026-09-10T21:07:19.522407+00:00
  cpu         Intel64 Family 6 Model 79 Stepping 1, GenuineIntel
  gpu_device  NVIDIA GeForce RTX 2060
  gpu_backend Vulkan
  platform    Windows-11-10.0.26100-SP0
  python      3.14.6
  reps       5  warmup 3  Ns [1000, 10000, 100000, 1000000, 500000]


`cpu_ms` is the CPU time; `warm_ms` is the GPU time after warm-up, so the
first-call shader-compile cost is not counted. `ratio > 1` means **the GPU is
slower**.

In [12]:
ops = ds["raw"]["ops"]
print(f"{'op':14s} {'N':>9s} {'sel':>7s} {'cpu_ms':>10s} {'gpu_ms':>10s} {'gpu/cpu':>8s}")
print("-" * 64)
for op in sorted(ops):
    for n_ in sorted(ops[op], key=int):
        for sel in sorted(ops[op][n_]):
            e = ops[op][n_][sel]
            cpu, gpu = e.get("cpu_ms"), e.get("warm_ms")
            if cpu is None or gpu is None:
                continue
            if n_ != max(ops[op], key=int):
                continue
            print(f"{op:14s} {n_:>9s} {sel:>7s} {cpu:10.4f} {gpu:10.4f} "
                  f"{gpu / cpu:7.1f}x")

op                     N     sel     cpu_ms     gpu_ms  gpu/cpu
----------------------------------------------------------------
compare          1000000     0.5     0.3499    17.5820    50.2x
filter           1000000     0.1     7.6000    41.4661     5.5x
filter           1000000     0.5    16.2128    45.9427     2.8x
filter           1000000     0.9    11.1611    51.3620     4.6x
gather           1000000     0.5     5.7639    21.5591     3.7x
groupby          1000000     100     2.1764    30.7358    14.1x
groupby          1000000   10000     5.2898    31.6800     6.0x
groupby          1000000  100000    27.5488    52.9420     1.9x
mask             1000000     0.5     0.3688    29.7893    80.8x
pack_keys        1000000     0.5    10.1482    24.9014     2.5x
reduce           1000000     0.5     0.9702     5.3880     5.6x
rng_fill_i32     1000000     0.5    60.6839    13.5651     0.2x
slice             100000     0.5     0.0236     5.8853   249.4x
sort             4194304     0.5   323.

Read the `gpu/cpu` column honestly. At the largest size each op was measured
at, **the GPU is slower than the CPU for eight of the ten operations** — by 3x
for `gather` and `pack_keys`, and by 25x to 80x for `filter`, `mask`, `compare`
and `slice`. Only two turn over at large N.

In [13]:
wins = {}
for op in sorted(ops):
    for n_ in sorted(ops[op], key=int):
        for sel in sorted(ops[op][n_]):
            e = ops[op][n_][sel]
            if e.get("cpu_ms") and e.get("warm_ms") and e["warm_ms"] < e["cpu_ms"]:
                wins.setdefault(op, []).append(int(n_))
print("ops where the measured GPU time is BELOW the CPU time:")
for op, sizes in wins.items():
    print(f"  {op:14s} at N = {sizes}")
print()
print("ops where it never is, in the sizes measured:")
print(" ", [op for op in sorted(ops) if op not in wins])

ops where the measured GPU time is BELOW the CPU time:
  rng_fill_i32   at N = [100000, 500000, 1000000]
  sort           at N = [4194304]

ops where it never is, in the sizes measured:
  ['compare', 'filter', 'gather', 'groupby', 'mask', 'pack_keys', 'reduce', 'slice']


### The crossover, stated exactly

* **`rng_fill_i32`**: GPU wins from N = 100,000 upward (and loses by 28x at
  N = 1,000). The crossing sits between 10,000 and 100,000.
* **`sort`**: GPU wins at N = 4,194,304, and loses by 105x at N = 8,192.
* **everything else**: the measured GPU time never drops below the CPU time at
  any measured size.

This is one GPU, one host, and one calibration run — not a property of the
engine. It is a property of *this measurement*, and it is quoted as such.

In [14]:
rng_ops = ops["rng_fill_i32"]
prev = None
for n_ in sorted(rng_ops, key=int):
    e = rng_ops[n_]["0.5"]
    verdict = "gpu faster" if e["warm_ms"] < e["cpu_ms"] else "cpu faster"
    print(f"  rng_fill_i32 N={int(n_):>9,}  cpu {e['cpu_ms']:8.4f} ms   "
          f"gpu {e['warm_ms']:8.4f} ms   -> {verdict}")

  rng_fill_i32 N=    1,000  cpu   0.0901 ms   gpu   2.5207 ms   -> cpu faster
  rng_fill_i32 N=   10,000  cpu   0.6737 ms   gpu   2.5298 ms   -> cpu faster
  rng_fill_i32 N=  100,000  cpu   6.1957 ms   gpu   2.7803 ms   -> gpu faster
  rng_fill_i32 N=  500,000  cpu  29.5588 ms   gpu   8.1786 ms   -> gpu faster
  rng_fill_i32 N=1,000,000  cpu  60.6839 ms   gpu  13.5651 ms   -> gpu faster


## 6. Why the GPU is behind, structurally

The recorded transfer costs say it without any argument. `gpu_execute` reads
each node's result back to the host, and on this machine a device-to-host copy
costs **more per byte than the host-to-device copy**, and both cost real
milliseconds. A single-op job therefore pays a round trip to compute one sum.

In [15]:
tr = ds["raw"]["transfer"]
for direction in ("h2d", "d2h"):
    print(direction.upper())
    for by in sorted(tr[direction], key=int):
        e = tr[direction][by]
        ns_per_byte = e["median_ms"] * 1e6 / e["bytes"]
        print(f"   {int(e['bytes']):>12,} B   {e['median_ms']:9.3f} ms   "
              f"{ns_per_byte:6.3f} ns/byte")

H2D
      1,048,576 B       0.899 ms    0.857 ns/byte
     10,485,760 B       7.506 ms    0.716 ns/byte
    104,857,600 B      82.252 ms    0.784 ns/byte
   1,073,741,824 B    2132.330 ms    1.986 ns/byte
D2H
      1,048,576 B       3.450 ms    3.290 ns/byte
     10,485,760 B      18.670 ms    1.781 ns/byte
    104,857,600 B     147.991 ms    1.411 ns/byte
   1,073,741,824 B    1438.620 ms    1.340 ns/byte


At 1 MiB a device-to-host copy costs about the same as this entire CPU
`filter` at N = 1,000. That is the crossover question answered without a
stopwatch: **the copy back is the cost, and the GPU cannot amortise it on a
single-op job.**

## 7. What this notebook does not claim

* No GPU speedup is claimed, and none is implied by anything above.
* No GPU residency is claimed. `gpu_execute` returns per-node buffers; there is
  no handle that stays on the device, and the surface carries no such verb —
  `test_the_surface_has_no_gpu_residency_verb` asserts `to_gpu`,
  `gpu_resident` and `stay_on_gpu` are absent from every reachable name.
* Through the consumer surface the GPU is unreachable at all.